# Component Classification — Colab

Runs `src/Object Detection/Component Training/` on a Colab GPU. Expects the dataset on Drive at `My Drive/FYP DATASET/component-symbols/` (folder with `manifest.jsonl`, `classes.txt`, `images/`, `labels/`).

Cells are safe to re-run in any order once setup has completed: the repo is pulled, not re-cloned, and the dataset copy is skipped when it is already in place.

## 1. Setup

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_SYMBOLS = Path("/content/drive/MyDrive/FYP DATASET/component-symbols")
assert (DRIVE_SYMBOLS / "manifest.jsonl").exists(), f"Not found: {DRIVE_SYMBOLS / 'manifest.jsonl'}"

In [ ]:
REPO = Path("/content/Single-Line-Diagram-and-Energy-Mapping")

# pull if already cloned; never delete the repo, the dataset copy lives inside it
if (REPO / ".git").exists():
    !git -C "{REPO}" pull -q
else:
    !git clone -q https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git "{REPO}"

TRAIN_DIR = REPO / "src" / "Object Detection" / "Component Training"
assert (TRAIN_DIR / "train.py").exists(), TRAIN_DIR

In [ ]:
import shutil

DEST = REPO / "component-symbols"

def count(root):
    return sum(1 for _ in (root / "images").rglob("*.png")) if (root / "images").exists() else -1

# Drive is slow to read; copy to local disk once, skip when the copy is already complete
if count(DEST) != count(DRIVE_SYMBOLS):
    shutil.rmtree(DEST, ignore_errors=True)
    shutil.copytree(DRIVE_SYMBOLS, DEST)

for must in ("manifest.jsonl", "classes.txt", "images/train", "images/val"):
    assert (DEST / must).exists(), f"Missing: {DEST / must}"

len(list((DEST / "images/train").glob("*.png"))), len(list((DEST / "images/val").glob("*.png")))

In [ ]:
%cd "{TRAIN_DIR}"
# Colab already ships tensorflow, scikit-learn, matplotlib; installing requirements.txt would reinstall TF
!pip install -q keras-tuner

import sys
import tensorflow as tf

sys.path.insert(0, str(TRAIN_DIR))
tf.__version__, tf.config.list_physical_devices("GPU")

## 2. Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from config import DEFAULT_HP, EPOCHS
from data import load_datasets
from model import build_geometric, build_model, compile_model, RandomDegrade
from train import callbacks, lr_schedule, setup
from evaluate import predict, most_confused

setup()
hp = DEFAULT_HP
train_ds, val_ds, class_names, class_weights = load_datasets(mixup_alpha=hp["mixup_alpha"])
len(class_names), int(train_ds.cardinality()), int(val_ds.cardinality())

In [ ]:
images, labels = next(iter(train_ds))
rows = [images, build_geometric()(images, training=True), RandomDegrade()(images, training=True)]

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for col in range(6):
    for r, batch in enumerate(rows):
        axes[r, col].imshow(batch[col, ..., 0], cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(class_names[int(np.argmax(labels[col]))], fontsize=8)
for ax in axes.flat:
    ax.axis("off")

## 3. Train

In [ ]:
model = build_model(len(class_names), hp)  # backbone="efficientnetv2b0" for the pretrained variant
compile_model(model, lr_schedule(int(train_ds.cardinality()), hp["learning_rate"]), hp)
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks("colab"),
)

In [ ]:
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
for key in ("accuracy", "val_accuracy", "top3", "val_top3"):
    ax_acc.plot(history.history[key], label=key)
for key in ("loss", "val_loss"):
    ax_loss.plot(history.history[key], label=key)
ax_acc.legend()
ax_loss.legend()

## 4. Validation: clean, test-time augmentation, scan-degraded

In [ ]:
results = {mode: predict(model, val_ds, mode) for mode in ("clean", "tta", "degraded")}
{mode: float(np.mean(y == p)) for mode, (y, p) in results.items()}

In [ ]:
labels, predictions = results["degraded"]  # the harder view is where confusions show
disp = ConfusionMatrixDisplay.from_predictions(labels, predictions, display_labels=class_names, xticks_rotation=90)
disp.figure_.set_size_inches(12, 12)

In [ ]:
print(classification_report(labels, predictions, target_names=class_names, zero_division=0))
most_confused(labels, predictions, class_names)

In [ ]:
model.save("models/colab.keras")

## 5. Optional: hyperparameter search and backbone comparison

These run the scripts as subprocesses from `TRAIN_DIR`. `tune.py` takes a while; compare models on the `degraded` metric in `outputs/<tag>_metrics.json`.

In [ ]:
!python tune.py
!python train.py --hp custom_best_hp.json
!python evaluate.py --model custom

In [ ]:
!python train.py --backbone efficientnetv2b0
!python evaluate.py --model efficientnetv2b0

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 6. Save results to Drive

Colab's local disk is wiped when the runtime ends; the models, metrics and logs go to `FYP DATASET/colab_runs/`.

In [ ]:
from datetime import datetime

run_dir = DRIVE_SYMBOLS.parent / "colab_runs" / datetime.now().strftime("%Y%m%d_%H%M")
for folder in ("models", "outputs", "logs"):
    if (TRAIN_DIR / folder).exists():
        shutil.copytree(TRAIN_DIR / folder, run_dir / folder, dirs_exist_ok=True)
run_dir